In [ ]:
# =====================================================
# 1. Library
# =====================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
import platform
import ast

pd.set_option("display.float_format", "{:.2f}".format)

if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"


# =====================================================
# 2. Data Load
# =====================================================
df = pd.read_csv("../원본 데이터 셋/2025_Airbnb_NYC_listings.csv")

print("데이터 크기:", df.shape)
df.head()


# =====================================================
# 3. Fake Null Check
# =====================================================
fake_null = ['NaN','None','none','NULL','null','N/A','na','']

fake_counts = df.isin(fake_null).sum()
print(fake_counts[fake_counts > 0])


# =====================================================
# 4. Datetime 변환
# =====================================================
date_cols = [
    "host_since",
    "first_review",
    "last_review",
    "calendar_last_scraped"
]

df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")


# =====================================================
# 5. Price Cleaning
# =====================================================
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("$","",regex=False)
    .str.replace(",","",regex=False)
    .astype(float)
)

df["price_log"] = np.log1p(df["price"])


# =====================================================
# 6. 결측치 처리
# =====================================================
review_cols = [
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "reviews_per_month",
    "estimated_occupancy_l365d"
]

df[review_cols] = df[review_cols].fillna(0)

df["review_scores_rating"] = df["review_scores_rating"].fillna(
    df["review_scores_rating"].median()
)


# =====================================================
# 7. Feature Engineering
# =====================================================
df["bathrooms"] = df["bathrooms"].fillna(0)
df["bedrooms"] = df["bedrooms"].fillna(0)
df["beds"] = df["beds"].fillna(0)

df["room_capacity"] = (
    df["bathrooms"] +
    df["bedrooms"] +
    df["beds"]
)

df["occupancy_rate"] = (
    df["estimated_occupancy_l365d"] / 365
)

df["log_reviews"] = np.log1p(df["number_of_reviews"])


# =====================================================
# 8. Amenities Count
# =====================================================
def amenity_count(x):

    try:
        return len(ast.literal_eval(x))
    except:
        return 0

df["amenities_len"] = df["amenities"].apply(amenity_count)


# =====================================================
# 9. 규제 기반 숙소 분류
# =====================================================
property_map = {

    "Room in hotel": "Lodging",
    "Room in boutique hotel": "Lodging",

    "Entire rental unit": "Residential",
    "Private room in rental unit": "Residential",
    "Private room in home": "Residential",
    "Entire home": "Residential",
    "Entire condo": "Residential"
}

df["property_regulation_type"] = (
    df["property_type"]
    .map(property_map)
    .fillna("Other")
)


# =====================================================
# 10. Legal Flag
# =====================================================
df["legal_flag"] = "Legal"

df.loc[
    (df["property_regulation_type"]=="Residential") &
    (df["room_type"]=="Entire home/apt") &
    (df["minimum_nights"] < 30),
    "legal_flag"
] = "Illegal"


# =====================================================
# 11. Strategy 분류
# =====================================================
def strategy(row):

    if row["property_regulation_type"]=="Residential":

        if row["minimum_nights"] < 30:
            return "Residential_short"

        else:
            return "Residential_mid"

    else:
        return "Hotel"

df["rental_strategy"] = df.apply(strategy, axis=1)


# =====================================================
# 12. 매출 계산 검증
# =====================================================
df["calc_revenue"] = (
    df["price"] *
    df["estimated_occupancy_l365d"]
)

print(
    "Revenue match:",
    (df["calc_revenue"]==df["estimated_revenue_l365d"]).mean()
)


# =====================================================
# 13. 가격 이상치 제거
# =====================================================
df = df[df["price"] < 1000]


# =====================================================
# 14. Price Distribution
# =====================================================
sns.histplot(df["price"], bins=50)
plt.title("Price Distribution")
plt.show()


# =====================================================
# 15. Price vs Occupancy
# =====================================================
sns.scatterplot(
    data=df,
    x="price",
    y="occupancy_rate",
    alpha=0.4
)

plt.title("Price vs Occupancy")
plt.show()


# =====================================================
# 16. Spearman Correlation
# =====================================================
corr, p = stats.spearmanr(df["price"], df["occupancy_rate"])

print("Spearman:", corr)
print("p:", p)


# =====================================================
# 17. Price Quantile Analysis
# =====================================================
df["price_q"] = pd.qcut(df["price"],4)

df.groupby("price_q")[
    ["occupancy_rate","estimated_revenue_l365d"]
].median()


# =====================================================
# 18. Review 기반 수요 분석
# =====================================================
sns.scatterplot(
    data=df,
    x="number_of_reviews",
    y="occupancy_rate"
)

plt.title("Reviews vs Occupancy")
plt.show()


# =====================================================
# 19. Availability 분석
# =====================================================
sns.boxplot(
    data=df,
    x="availability_365",
    y="price"
)

plt.title("Availability vs Price")
plt.show()


# =====================================================
# 20. 매출 0 숙소 분석
# =====================================================
zero_ratio = (
    df["estimated_revenue_l365d"]==0
).mean()

print("Revenue zero ratio:",zero_ratio)


# =====================================================
# 21. Kruskal Test
# =====================================================
groups = [

df[df["rental_strategy"]=="Residential_mid"]
["estimated_revenue_l365d"],

df[df["rental_strategy"]=="Residential_short"]
["estimated_revenue_l365d"],

df[df["rental_strategy"]=="Hotel"]
["estimated_revenue_l365d"]

]

print(stats.kruskal(*groups))


# =====================================================
# 22. ANOVA
# =====================================================
import statsmodels.api as sm
from statsmodels.formula.api import ols

anova_model = ols(

"estimated_revenue_l365d ~ C(rental_strategy)",

data=df

).fit()

print(sm.stats.anova_lm(anova_model))


# =====================================================
# 23. 회귀 분석
# =====================================================
model = smf.ols(

"""occupancy_rate ~

price_log
+ log_reviews
+ review_scores_rating
+ accommodates
+ minimum_nights
+ amenities_len
""",

data=df

).fit()

print(model.summary())


# =====================================================
# 24. Regression Visualization
# =====================================================
coef = model.params.drop("Intercept")

coef.sort_values().plot(
kind="barh",
figsize=(8,6)
)

plt.title("Occupancy Drivers")
plt.show()


# =====================================================
# 25. Market Heatmap
# =====================================================
heat = df.pivot_table(

values="estimated_revenue_l365d",
index="neighbourhood_group_cleansed",
columns="room_type",
aggfunc="median"

)

sns.heatmap(
heat,
annot=True,
fmt=".0f",
cmap="YlOrRd"
)

plt.title("Revenue Heatmap")
plt.show()


# =====================================================
# 26. 투자 시뮬레이터
# =====================================================
def simulate_investment(
borough,
room_type,
investment=500000
):

subset = df[
(df["neighbourhood_group_cleansed"]==borough) &
(df["room_type"]==room_type)
]

price = subset["price"].median()
occ = subset["occupancy_rate"].median()

annual = price*occ*365
monthly = annual/12

cost_ratio=0.30

profit=monthly*(1-cost_ratio)

payback=investment/(profit*12)

return {

"median_price":price,
"occupancy_rate":occ,
"monthly_revenue":monthly,
"monthly_profit":profit,
"payback_year":payback

}


# =====================================================
# 27. Example
# =====================================================
simulate_investment(
"Manhattan",
"Entire home/apt"
)

In [ ]:
# =====================================================
# 1. Library
# =====================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.formula.api as smf
import platform
import ast

pd.set_option("display.float_format", "{:.2f}".format)

if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"


# =====================================================
# 2. Data Load
# =====================================================
df = pd.read_csv("../원본 데이터 셋/2025_Airbnb_NYC_listings.csv")

print("데이터 크기:", df.shape)
df.head()


# =====================================================
# 3. Data Overview
# =====================================================
df.info()
df.describe()


# =====================================================
# 4. Data Cleaning
# =====================================================

# Fake null 확인
fake_null = ['NaN','None','none','NULL','null','N/A','na','']

fake_counts = df.isin(fake_null).sum()
print(fake_counts[fake_counts > 0])

# 날짜 변환
date_cols = [
"host_since",
"first_review",
"last_review",
"calendar_last_scraped"
]

df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")


# 가격 타입 변환
df["price"] = (
df["price"]
.astype(str)
.str.replace("$","",regex=False)
.str.replace(",","",regex=False)
.astype(float)
)

df["price_log"] = np.log1p(df["price"])


# =====================================================
# 5. Feature Engineering
# =====================================================

df["bathrooms"] = df["bathrooms"].fillna(0)
df["bedrooms"] = df["bedrooms"].fillna(0)
df["beds"] = df["beds"].fillna(0)

df["room_capacity"] = (
df["bathrooms"] +
df["bedrooms"] +
df["beds"]
)

df["occupancy_rate"] = (
df["estimated_occupancy_l365d"] / 365
)

df["log_reviews"] = np.log1p(df["number_of_reviews"])


# amenities 수
def amenity_count(x):
    try:
        return len(ast.literal_eval(x))
    except:
        return 0

df["amenities_len"] = df["amenities"].apply(amenity_count)


# =====================================================
# 6. NYC Regulation Classification
# =====================================================

property_map = {

"Room in hotel":"Lodging",
"Room in boutique hotel":"Lodging",

"Entire rental unit":"Residential",
"Private room in rental unit":"Residential",
"Private room in home":"Residential",
"Entire home":"Residential",
"Entire condo":"Residential"

}

df["property_regulation_type"] = (
df["property_type"]
.map(property_map)
.fillna("Other")
)


# 합법 여부
df["legal_flag"] = "Legal"

df.loc[
(df["property_regulation_type"]=="Residential") &
(df["room_type"]=="Entire home/apt") &
(df["minimum_nights"] < 30),
"legal_flag"
] = "Illegal"


# =====================================================
# 7. Strategy Classification
# =====================================================

def strategy(row):

    if row["property_regulation_type"]=="Residential":

        if row["minimum_nights"] < 30:
            return "Residential_short_term"

        else:
            return "Residential_long_term"

    else:
        return "Hotel"

df["rental_strategy"] = df.apply(strategy, axis=1)


# =====================================================
# 8. Exploratory Data Analysis
# =====================================================

# 가격 분포
sns.histplot(df["price"], bins=50)
plt.title("Price Distribution")
plt.show()

# borough별 공급
sns.countplot(data=df, x="neighbourhood_group_cleansed")
plt.title("Listing Count by Borough")
plt.show()


# =====================================================
# 9. Demand Analysis
# =====================================================

# 가격 vs 점유율
sns.scatterplot(
data=df,
x="price",
y="occupancy_rate",
alpha=0.4
)

plt.title("Price vs Occupancy")
plt.show()


# 리뷰 vs 점유율
sns.scatterplot(
data=df,
x="number_of_reviews",
y="occupancy_rate"
)

plt.title("Reviews vs Occupancy")
plt.show()


# =====================================================
# 10. Statistical Test
# =====================================================

corr, p = stats.spearmanr(
df["price"],
df["occupancy_rate"]
)

print("Spearman correlation:", corr)
print("p-value:", p)


# =====================================================
# 11. Regression Analysis
# =====================================================

model = smf.ols(

"""
occupancy_rate ~

price_log
+ log_reviews
+ review_scores_rating
+ accommodates
+ minimum_nights
+ amenities_len
"""

,data=df

).fit()

print(model.summary())


# 회귀 계수 시각화
coef = model.params.drop("Intercept")

coef.sort_values().plot(
kind="barh",
figsize=(8,6)
)

plt.title("Occupancy Drivers")
plt.show()


# =====================================================
# 12. Market Structure Analysis
# =====================================================

heat = df.pivot_table(

values="estimated_revenue_l365d",
index="neighbourhood_group_cleansed",
columns="room_type",
aggfunc="median"

)

sns.heatmap(
heat,
annot=True,
fmt=".0f",
cmap="YlOrRd"
)

plt.title("Revenue Heatmap")
plt.show()


# =====================================================
# 13. Persona Simulation
# =====================================================

def simulate_investment(
borough,
room_type,
investment=500000
):

subset = df[
(df["neighbourhood_group_cleansed"]==borough) &
(df["room_type"]==room_type)
]

price = subset["price"].median()
occ = subset["occupancy_rate"].median()

annual_revenue = price * occ * 365
monthly_revenue = annual_revenue / 12

cost_ratio = 0.3
monthly_profit = monthly_revenue * (1-cost_ratio)

payback_year = investment / (monthly_profit*12)

return {

"median_price":price,
"occupancy_rate":occ,
"monthly_revenue":monthly_revenue,
"monthly_profit":monthly_profit,
"payback_year":payback_year

}


# =====================================================
# 14. 예시 시나리오
# =====================================================


# 14. 예시 시나리오
| 항목 | 시나리오 1: 브루클린 | 시나리오 2: 맨해튼 | 시나리오 3: 퀸즈 (최적안) |
|-----|------------------|------------------|------------------|
| 페르소나 | 단체 손님 전문 펜션지기 | 리스크 관리 전략가 | 3개월 컷 프로 부업러 |
| 투자 원금 | $25,000 (약 3,300만 원) | $30,000 (약 4,000만 원) | $15,000 (약 2,000만 원) |
| 숙소 유형 | Entire home (독채 전체) | Private room (개인실) | Private room (개인실) |
| 핵심 타겟 | 14인 이상 대규모 단체 | 5인 가족 및 비즈니스 팀 | 6인 우정 여행 / 소가족 |
| 평균 가격 | $374.00 | $385.50 | $269.00 |
| 시장 포지션 | 초블루오션 (숙소 5개뿐) | 블루오션 (숙소 4개뿐) | 블루오션 (숙소 8개뿐) |
| 월 순이익 | $2,766.58 | $3,029.87 | $4,228.46 |
| 회수 기간 | 9.04개월 | 9.9개월 | **3.55개월 (압도적)** |
| 한 줄 전략 | "레드오션 독채들 사이에서 '대형 인원'으로만 승부한다." | "비싼 맨해튼에서 1인실 버리고 5인실로 단가 싸움한다." | "가장 적은 돈 투자해서 가장 빨리 뽑고 돈 복사 시작한다." |